# Domain Adaptation & Unpaired Image-to-Image Translation using CycleGAN

**Objective:** Design and implement an **unpaired** image-to-image translation system using
**CycleGAN** to learn bidirectional mappings between two visual domains **without** paired data.

| Direction | Generator | Purpose |
|-----------|-----------|---------|
| Sketch → Photo | G_AB | Translate sketches into realistic photos |
| Photo → Sketch | G_BA | Translate photos into sketch-style drawings |

Structural consistency is enforced using **cycle consistency** — translating an image to the other
domain and back should reconstruct the original.

## Architecture Overview

| Component | Architecture | Details |
|-----------|-------------|---------|
| Generators (×2) | **ResNet** (6 blocks) | Encoder → Residual blocks → Decoder |
| Discriminators (×2) | **PatchGAN** | Patch-level real/fake classification |
| Adversarial Loss | LSGAN (MSE) | Smoother gradients than BCE |
| Cycle Loss | L1 | \|G_BA(G_AB(x)) - x\| + \|G_AB(G_BA(y)) - y\| |
| Identity Loss | L1 | Colour/tone preservation |

**Dataset:** Sketchy Dataset (sketches + photos by category)  
**Platform:** Kaggle GPU T4 × 2  
**Image Size:** 128 × 128

In [ ]:
# ============================================================
# 1. Environment Setup
# ============================================================
!pip install -q gradio scikit-image

import os, sys, random, time, glob, itertools, warnings, copy
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as T
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from collections import deque

import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  ({p.total_mem / 1024**3:.1f} GB)")

In [ ]:
# ============================================================
# 2. Configuration & Hyperparameters
# ============================================================

# ── Dataset paths (Kaggle) ─────────────────────────────────────
# Sketchy dataset: contains /sketch/ and /photo/ folders with category subfolders
SKETCHY_PATH = "/kaggle/input/sketchy-dataset"

# Domain A = Sketch,  Domain B = Photo
DOMAIN_A_PATH = None   # Auto-detected below
DOMAIN_B_PATH = None   # Auto-detected below

# ── Architecture ───────────────────────────────────────────────
IMAGE_SIZE    = 128       # 128×128 (reduced for Kaggle memory)
NC            = 3         # RGB channels
N_RES_BLOCKS  = 6         # ResNet blocks in generator (6 for Kaggle, 9 for full)

# ── Training ───────────────────────────────────────────────────
BATCH_SIZE    = 4         # Small batch for CycleGAN memory
NUM_EPOCHS    = 40        # Adjust based on GPU quota
LR            = 0.0002
BETA1         = 0.5
BETA2         = 0.999
LAMBDA_CYCLE  = 10.0      # Cycle consistency loss weight
LAMBDA_IDENT  = 5.0       # Identity loss weight (0.5 * LAMBDA_CYCLE)
MAX_SAMPLES   = 3000      # Use a subset per domain (speed + memory)

# ── Mixed Precision ───────────────────────────────────────────
USE_AMP       = True

# ── Replay Buffer ─────────────────────────────────────────────
BUFFER_SIZE   = 50        # Image replay buffer for discriminator stability

# ── Checkpointing ─────────────────────────────────────────────
SAVE_EVERY    = 5
OUTPUT_DIR    = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── LR Scheduling ─────────────────────────────────────────────
DECAY_START   = NUM_EPOCHS // 2   # Start linear LR decay at halfway

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"GPUs          : {torch.cuda.device_count()}")
print(f"Image size    : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"ResNet blocks : {N_RES_BLOCKS}")
print(f"λ_cycle       : {LAMBDA_CYCLE}")
print(f"λ_identity    : {LAMBDA_IDENT}")

# 3. Data Preparation

CycleGAN uses **unpaired** data — we simply need a collection of images from Domain A (sketches)
and a separate collection from Domain B (photos). There is **no** requirement for 1-to-1 correspondence.

We auto-detect the Sketchy dataset structure which typically contains `sketch/` and `photo/`
subdirectories with category subfolders inside each.

In [ ]:
# ============================================================
# 3. Data Preparation
# ============================================================

# ── Auto-detect dataset structure ──
def find_domain_dirs(root):
    """Find sketch and photo directories in the Sketchy dataset."""
    sketch_dir, photo_dir = None, None
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            dl = d.lower()
            if 'sketch' in dl and sketch_dir is None:
                candidate = os.path.join(dirpath, d)
                # Check it actually contains images or category subdirs
                sketch_dir = candidate
            elif 'photo' in dl and photo_dir is None:
                candidate = os.path.join(dirpath, d)
                photo_dir = candidate
        if sketch_dir and photo_dir:
            break
    return sketch_dir, photo_dir

DOMAIN_A_PATH, DOMAIN_B_PATH = find_domain_dirs(SKETCHY_PATH)

if DOMAIN_A_PATH is None or DOMAIN_B_PATH is None:
    # Fallback: list top-level contents for debugging
    print(f"Could not auto-detect. Contents of {SKETCHY_PATH}:")
    for item in os.listdir(SKETCHY_PATH):
        full = os.path.join(SKETCHY_PATH, item)
        if os.path.isdir(full):
            n = sum(1 for _ in os.scandir(full))
            print(f"  📁 {item}/ ({n} items)")
        else:
            print(f"  📄 {item}")
    raise FileNotFoundError(
        "Please set DOMAIN_A_PATH and DOMAIN_B_PATH manually above."
    )

print(f"Domain A (Sketch): {DOMAIN_A_PATH}")
print(f"Domain B (Photo) : {DOMAIN_B_PATH}")

In [ ]:
class UnpairedDomainDataset(data.Dataset):
    """Load images from a single domain directory (recursively)."""
    EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

    def __init__(self, root, size=128, max_samples=None):
        self.size = size
        self.paths = []
        for dp, _, fns in os.walk(root):
            for f in sorted(fns):
                if os.path.splitext(f)[1].lower() in self.EXTS:
                    self.paths.append(os.path.join(dp, f))

        # Optionally limit dataset size
        if max_samples and len(self.paths) > max_samples:
            random.shuffle(self.paths)
            self.paths = self.paths[:max_samples]
            self.paths.sort()

        self.transform = T.Compose([
            T.Resize((size, size)),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])
        print(f"  Loaded {len(self.paths)} images from {root}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img)


print("Loading Domain A (Sketches)...")
ds_A = UnpairedDomainDataset(DOMAIN_A_PATH, IMAGE_SIZE, MAX_SAMPLES)
print("Loading Domain B (Photos)...")
ds_B = UnpairedDomainDataset(DOMAIN_B_PATH, IMAGE_SIZE, MAX_SAMPLES)

loader_A = data.DataLoader(ds_A, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)
loader_B = data.DataLoader(ds_B, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)

print(f"\nDomain A batches: {len(loader_A)}")
print(f"Domain B batches: {len(loader_B)}")

In [ ]:
# ── Visualise samples from each domain ──
fig, axes = plt.subplots(2, 6, figsize=(18, 6))

sample_A = next(iter(loader_A))
sample_B = next(iter(loader_B))

for j in range(6):
    # Domain A
    img = sample_A[j].permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[0, j].imshow(img.clip(0, 1))
    axes[0, j].set_title("Sketch (A)", fontsize=9)
    axes[0, j].axis("off")
    # Domain B
    if j < sample_B.size(0):
        img = sample_B[j].permute(1, 2, 0).numpy() * 0.5 + 0.5
        axes[1, j].imshow(img.clip(0, 1))
    axes[1, j].set_title("Photo (B)", fontsize=9)
    axes[1, j].axis("off")

plt.suptitle("Unpaired Samples from Each Domain", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 4. Model Architecture

## 4.1 ResNet-Based Generator

Unlike the U-Net used in Pix2Pix, CycleGAN uses a **ResNet generator** which is better suited
for style transfer tasks where we want to change texture/appearance while preserving global structure.

```
Input (3×128×128)
  ↓ Conv 7×7 → 64 channels  (reflect padding)
  ↓ Conv 3×3 stride 2 → 128  (downsample)
  ↓ Conv 3×3 stride 2 → 256  (downsample)
  ↓ 6× ResNet Blocks (256 ch)
  ↓ ConvTranspose 3×3 → 128  (upsample)
  ↓ ConvTranspose 3×3 → 64   (upsample)
  ↓ Conv 7×7 → 3 channels + Tanh
Output (3×128×128)
```

## 4.2 PatchGAN Discriminator
Same concept as Pix2Pix, but operates on a **single** image (no concatenation with input)
since CycleGAN is unpaired.

In [ ]:
# ============================================================
# 4. Model Definitions
# ============================================================

def weights_init(m):
    cn = m.__class__.__name__
    if 'Conv' in cn and hasattr(m, 'weight'):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cn or 'InstanceNorm' in cn:
        if m.weight is not None:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)


# ────────────────────── Residual Block ──────────────────────

class ResidualBlock(nn.Module):
    """A single residual block with two 3×3 convolutions and instance norm."""
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3, bias=False),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3, bias=False),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)


# ────────────────────── ResNet Generator ──────────────────────

class ResNetGenerator(nn.Module):
    """
    ResNet-based generator for CycleGAN.
    Architecture: c7s1-64, d128, d256, R256×n_blocks, u128, u64, c7s1-3
    """
    def __init__(self, in_ch=NC, out_ch=NC, n_blocks=N_RES_BLOCKS, ngf=64):
        super().__init__()

        # ── Initial convolution block ──
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, ngf, 7, bias=False),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True),
        ]

        # ── Downsampling (encoder) ──
        in_features = ngf
        for _ in range(2):
            out_features = in_features * 2
            model += [
                nn.Conv2d(in_features, out_features, 3, stride=2, padding=1, bias=False),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True),
            ]
            in_features = out_features

        # ── Residual blocks ──
        for _ in range(n_blocks):
            model += [ResidualBlock(in_features)]

        # ── Upsampling (decoder) ──
        for _ in range(2):
            out_features = in_features // 2
            model += [
                nn.ConvTranspose2d(in_features, out_features, 3,
                                   stride=2, padding=1, output_padding=1, bias=False),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True),
            ]
            in_features = out_features

        # ── Output layer ──
        model += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, out_ch, 7),
            nn.Tanh(),
        ]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


# ────────────────────── PatchGAN Discriminator ──────────────────────

class PatchGANDiscriminator(nn.Module):
    """
    PatchGAN discriminator for a single image (not concatenated).
    Uses InstanceNorm (better for CycleGAN than BatchNorm).
    """
    def __init__(self, in_ch=NC, ndf=64):
        super().__init__()
        self.model = nn.Sequential(
            # Layer 1: no norm
            nn.Conv2d(in_ch, ndf, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 2
            nn.Conv2d(ndf, ndf * 2, 4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 3
            nn.Conv2d(ndf * 2, ndf * 4, 4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 4: stride 1
            nn.Conv2d(ndf * 4, ndf * 8, 4, stride=1, padding=1, bias=False),
            nn.InstanceNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # Output: patch classification
            nn.Conv2d(ndf * 8, 1, 4, stride=1, padding=1),
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# ── Instantiate all four networks ──
G_AB = ResNetGenerator().to(device)   # Sketch → Photo
G_BA = ResNetGenerator().to(device)   # Photo → Sketch
D_A  = PatchGANDiscriminator().to(device)  # Classifies sketches (domain A)
D_B  = PatchGANDiscriminator().to(device)  # Classifies photos  (domain B)

# Multi-GPU
if torch.cuda.device_count() > 1:
    print(f"Wrapping models with DataParallel ({torch.cuda.device_count()} GPUs)")
    G_AB = nn.DataParallel(G_AB)
    G_BA = nn.DataParallel(G_BA)
    D_A  = nn.DataParallel(D_A)
    D_B  = nn.DataParallel(D_B)

# Weight init
for net in [G_AB, G_BA, D_A, D_B]:
    net.apply(weights_init)

# Print summaries
print("=" * 60)
print("RESNET GENERATOR (×2: G_AB and G_BA)")
print("=" * 60)
print(G_AB)
g_params = sum(p.numel() for p in G_AB.parameters())
print(f"\nParameters per generator: {g_params:,}")
print(f"Total generator params : {g_params * 2:,}")

print("\n" + "=" * 60)
print("PATCHGAN DISCRIMINATOR (×2: D_A and D_B)")
print("=" * 60)
print(D_A)
d_params = sum(p.numel() for p in D_A.parameters())
print(f"\nParameters per discriminator: {d_params:,}")
print(f"Total discriminator params  : {d_params * 2:,}")
print(f"\nGrand total: {(g_params * 2 + d_params * 2):,} parameters")

# Verify shapes
with torch.no_grad():
    dummy = torch.randn(1, NC, IMAGE_SIZE, IMAGE_SIZE, device=device)
    out_g = G_AB(dummy)
    out_d = D_A(dummy)
    print(f"\nGenerator   : {list(dummy.shape)} → {list(out_g.shape)}")
    print(f"Discriminator: {list(dummy.shape)} → {list(out_d.shape)}")

# 5. Training

## Loss Functions

CycleGAN optimises three losses simultaneously:

### 1. Adversarial Loss (LSGAN — MSE)
Encourages G_AB to produce images that D_B classifies as real photos, and vice versa.
Using MSE (least-squares) instead of BCE for smoother gradients:

$$\mathcal{L}_{adv}(G_{AB}, D_B) = \mathbb{E}\left[(D_B(G_{AB}(a)) - 1)^2\right]$$

### 2. Cycle Consistency Loss (L1)
Translating to the other domain and back should reconstruct the original:

$$\mathcal{L}_{cycle} = \lambda_{cyc} \left( \| G_{BA}(G_{AB}(a)) - a \|_1 + \| G_{AB}(G_{BA}(b)) - b \|_1 \right)$$

### 3. Identity Loss (L1)
If we feed a photo to G_AB (which should output a photo), it should not change it:

$$\mathcal{L}_{ident} = \lambda_{id} \left( \| G_{AB}(b) - b \|_1 + \| G_{BA}(a) - a \|_1 \right)$$

In [ ]:
# ============================================================
# 5. Training Setup
# ============================================================

# ── Losses ──
criterion_GAN   = nn.MSELoss()    # LSGAN
criterion_cycle = nn.L1Loss()
criterion_ident = nn.L1Loss()

# ── Optimizers (joint for both generators, separate for each discriminator) ──
opt_G = optim.Adam(
    itertools.chain(G_AB.parameters(), G_BA.parameters()),
    lr=LR, betas=(BETA1, BETA2)
)
opt_D_A = optim.Adam(D_A.parameters(), lr=LR, betas=(BETA1, BETA2))
opt_D_B = optim.Adam(D_B.parameters(), lr=LR, betas=(BETA1, BETA2))

# ── Learning Rate Scheduler (linear decay after DECAY_START) ──
def lr_lambda(epoch):
    if epoch < DECAY_START:
        return 1.0
    return 1.0 - (epoch - DECAY_START) / (NUM_EPOCHS - DECAY_START + 1)

sched_G   = optim.lr_scheduler.LambdaLR(opt_G,   lr_lambda=lr_lambda)
sched_D_A = optim.lr_scheduler.LambdaLR(opt_D_A, lr_lambda=lr_lambda)
sched_D_B = optim.lr_scheduler.LambdaLR(opt_D_B, lr_lambda=lr_lambda)

# ── Mixed Precision Scalers ──
scaler_g  = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler_da = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler_db = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# ── Image Replay Buffer ──
class ReplayBuffer:
    """Stores previously generated images and randomly returns old or new ones.
    Helps stabilise discriminator training (Shrivastava et al., 2017)."""
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, images):
        result = []
        for img in images:
            img = img.unsqueeze(0)
            if len(self.data) < self.max_size:
                self.data.append(img.clone())
                result.append(img)
            elif random.random() > 0.5:
                idx = random.randint(0, self.max_size - 1)
                old = self.data[idx].clone()
                self.data[idx] = img.clone()
                result.append(old)
            else:
                result.append(img)
        return torch.cat(result, dim=0)

buffer_fake_A = ReplayBuffer(BUFFER_SIZE)
buffer_fake_B = ReplayBuffer(BUFFER_SIZE)

def gpu_mem():
    if not torch.cuda.is_available():
        return
    for i in range(torch.cuda.device_count()):
        a = torch.cuda.memory_allocated(i) / 1024**2
        r = torch.cuda.memory_reserved(i) / 1024**2
        print(f"  GPU {i}: {a:.0f} MB alloc / {r:.0f} MB reserved")

print("Training configuration ready.")
gpu_mem()

In [ ]:
# ============================================================
# Training Loop
# ============================================================

hist = {
    "G_total": [], "G_adv": [], "G_cycle": [], "G_ident": [],
    "D_A": [], "D_B": [],
    "G_epoch": [], "DA_epoch": [], "DB_epoch": [], "cycle_epoch": [],
}
# Visual snapshots: store (real_A, fake_B, recon_A, real_B, fake_A, recon_B)
snapshots = []

print(f"Starting CycleGAN training for {NUM_EPOCHS} epochs...")
t0 = time.time()

for epoch in range(NUM_EPOCHS):
    G_AB.train(); G_BA.train(); D_A.train(); D_B.train()
    ep_g, ep_da, ep_db, ep_cyc, n = 0., 0., 0., 0., 0

    for i, (real_A, real_B) in enumerate(zip(loader_A, loader_B)):
        real_A = real_A.to(device)
        real_B = real_B.to(device)
        bs = real_A.size(0)

        # ═══════════════════════════════════════════════════
        # (1) Update Generators (G_AB and G_BA)
        # ═══════════════════════════════════════════════════
        opt_G.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            # ── Identity loss ──
            ident_B = G_AB(real_B)  # G_AB should be identity on domain B
            ident_A = G_BA(real_A)  # G_BA should be identity on domain A
            loss_ident = (criterion_ident(ident_B, real_B) +
                          criterion_ident(ident_A, real_A)) * LAMBDA_IDENT

            # ── Adversarial loss ──
            fake_B = G_AB(real_A)   # Sketch → Photo
            pred_fake_B = D_B(fake_B)
            target_real = torch.ones_like(pred_fake_B, device=device)
            loss_G_AB = criterion_GAN(pred_fake_B, target_real)

            fake_A = G_BA(real_B)   # Photo → Sketch
            pred_fake_A = D_A(fake_A)
            target_real = torch.ones_like(pred_fake_A, device=device)
            loss_G_BA = criterion_GAN(pred_fake_A, target_real)

            loss_adv = loss_G_AB + loss_G_BA

            # ── Cycle consistency loss ──
            recon_A = G_BA(fake_B)  # Sketch → Photo → Sketch
            recon_B = G_AB(fake_A)  # Photo → Sketch → Photo
            loss_cycle = (criterion_cycle(recon_A, real_A) +
                          criterion_cycle(recon_B, real_B)) * LAMBDA_CYCLE

            # ── Total generator loss ──
            loss_G = loss_adv + loss_cycle + loss_ident

        scaler_g.scale(loss_G).backward()
        scaler_g.step(opt_G)
        scaler_g.update()

        # ═══════════════════════════════════════════════════
        # (2) Update Discriminator D_A (sketches)
        # ═══════════════════════════════════════════════════
        opt_D_A.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred_real = D_A(real_A)
            target_real = torch.ones_like(pred_real, device=device)
            loss_real = criterion_GAN(pred_real, target_real)

            fake_A_buf = buffer_fake_A.push_and_pop(fake_A.detach())
            pred_fake = D_A(fake_A_buf)
            target_fake = torch.zeros_like(pred_fake, device=device)
            loss_fake = criterion_GAN(pred_fake, target_fake)

            loss_DA = (loss_real + loss_fake) * 0.5

        scaler_da.scale(loss_DA).backward()
        scaler_da.step(opt_D_A)
        scaler_da.update()

        # ═══════════════════════════════════════════════════
        # (3) Update Discriminator D_B (photos)
        # ═══════════════════════════════════════════════════
        opt_D_B.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred_real = D_B(real_B)
            target_real = torch.ones_like(pred_real, device=device)
            loss_real = criterion_GAN(pred_real, target_real)

            fake_B_buf = buffer_fake_B.push_and_pop(fake_B.detach())
            pred_fake = D_B(fake_B_buf)
            target_fake = torch.zeros_like(pred_fake, device=device)
            loss_fake = criterion_GAN(pred_fake, target_fake)

            loss_DB = (loss_real + loss_fake) * 0.5

        scaler_db.scale(loss_DB).backward()
        scaler_db.step(opt_D_B)
        scaler_db.update()

        # ── Logging ──
        hist["G_total"].append(loss_G.item())
        hist["G_adv"].append(loss_adv.item())
        hist["G_cycle"].append(loss_cycle.item())
        hist["G_ident"].append(loss_ident.item())
        hist["D_A"].append(loss_DA.item())
        hist["D_B"].append(loss_DB.item())
        ep_g += loss_G.item(); ep_da += loss_DA.item()
        ep_db += loss_DB.item(); ep_cyc += loss_cycle.item()
        n += 1

        if i % 100 == 0:
            print(f"  [{epoch+1}/{NUM_EPOCHS}][{i}]  "
                  f"G: {loss_G.item():.3f} (adv={loss_adv.item():.3f}, "
                  f"cyc={loss_cycle.item():.3f}, id={loss_ident.item():.3f})  "
                  f"D_A: {loss_DA.item():.3f}  D_B: {loss_DB.item():.3f}")

    hist["G_epoch"].append(ep_g / n)
    hist["DA_epoch"].append(ep_da / n)
    hist["DB_epoch"].append(ep_db / n)
    hist["cycle_epoch"].append(ep_cyc / n)

    # ── LR step ──
    sched_G.step(); sched_D_A.step(); sched_D_B.step()
    cur_lr = opt_G.param_groups[0]['lr']

    # ── Snapshot ──
    G_AB.eval(); G_BA.eval()
    with torch.no_grad():
        s_A = next(iter(loader_A))[:4].to(device)
        s_B = next(iter(loader_B))[:4].to(device)
        s_fB = G_AB(s_A)
        s_rA = G_BA(s_fB)
        s_fA = G_BA(s_B)
        s_rB = G_AB(s_fA)
        snapshots.append((
            s_A.cpu(), s_fB.cpu(), s_rA.cpu(),
            s_B.cpu(), s_fA.cpu(), s_rB.cpu()
        ))

    # ── Checkpoint ──
    if (epoch + 1) % SAVE_EVERY == 0:
        for name, net in [("G_AB", G_AB), ("G_BA", G_BA), ("D_A", D_A), ("D_B", D_B)]:
            torch.save(net.state_dict(),
                       os.path.join(OUTPUT_DIR, f"cyclegan_{name}_ep{epoch+1}.pth"))
        print(f"  ✓ Checkpoint saved (epoch {epoch+1}, lr={cur_lr:.6f})")
        gpu_mem()

elapsed = time.time() - t0
for name, net in [("G_AB", G_AB), ("G_BA", G_BA), ("D_A", D_A), ("D_B", D_B)]:
    torch.save(net.state_dict(), os.path.join(OUTPUT_DIR, f"cyclegan_{name}_final.pth"))
print(f"\n✔ CycleGAN training complete — {elapsed/60:.1f} min")

# 6. Visualisation & Evaluation

## 6.1 Training Loss Curves

In [ ]:
# ============================================================
# 6.1 Loss Curves
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── Epoch-Averaged: G vs D ──
ax = axes[0, 0]
ax.set_title("Generator vs Discriminator (Epoch Avg)", fontsize=12)
ax.plot(hist["G_epoch"],  marker="o", label="G total", color="tab:blue")
ax.plot(hist["DA_epoch"], marker="s", label="D_A", color="tab:orange")
ax.plot(hist["DB_epoch"], marker="^", label="D_B", color="tab:red")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend()

# ── Cycle Consistency Loss ──
ax = axes[0, 1]
ax.set_title("Cycle Consistency Loss (Epoch Avg)", fontsize=12)
ax.plot(hist["cycle_epoch"], marker="o", color="tab:green", linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Cycle Loss")

# ── Generator Components (per-iteration, smoothed) ──
ax = axes[1, 0]
ax.set_title("Generator Loss Components (per iteration)", fontsize=12)
ax.plot(hist["G_adv"],   alpha=0.2, color="tab:blue")
ax.plot(hist["G_cycle"], alpha=0.2, color="tab:green")
ax.plot(hist["G_ident"], alpha=0.2, color="tab:purple")
# Smoothed
win = min(50, len(hist["G_adv"]) // 2)
if win > 1:
    kernel = np.ones(win) / win
    ax.plot(np.convolve(hist["G_adv"], kernel, 'valid'),
            color="tab:blue", lw=2, label="Adversarial")
    ax.plot(np.convolve(hist["G_cycle"], kernel, 'valid'),
            color="tab:green", lw=2, label="Cycle")
    ax.plot(np.convolve(hist["G_ident"], kernel, 'valid'),
            color="tab:purple", lw=2, label="Identity")
ax.set_xlabel("Iteration"); ax.set_ylabel("Loss"); ax.legend()

# ── Discriminator per-iteration ──
ax = axes[1, 1]
ax.set_title("Discriminator Losses (per iteration)", fontsize=12)
ax.plot(hist["D_A"], alpha=0.2, color="tab:orange")
ax.plot(hist["D_B"], alpha=0.2, color="tab:red")
if win > 1:
    ax.plot(np.convolve(hist["D_A"], kernel, 'valid'),
            color="tab:orange", lw=2, label="D_A (sketch)")
    ax.plot(np.convolve(hist["D_B"], kernel, 'valid'),
            color="tab:red", lw=2, label="D_B (photo)")
ax.set_xlabel("Iteration"); ax.set_ylabel("Loss"); ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curves.png"), dpi=150)
plt.show()

## 6.2 Translation Results — Cycle Visualisation

For each direction we show:
- **Input** → **Translated** → **Reconstructed** (cycle)

If the cycle loss is working, the reconstructed image should closely match the original input.

In [ ]:
# ============================================================
# 6.2 Cycle Translation Visualisation
# ============================================================

def show_cycle(real, fake, recon, direction, n=5):
    """Display Input → Translated → Reconstructed for n samples."""
    n = min(n, real.size(0))
    fig, axes = plt.subplots(3, n, figsize=(3.5 * n, 10))
    labels = ["Input", "Translated", "Reconstructed"]

    for col in range(n):
        for row, tensor in enumerate([real, fake, recon]):
            img = tensor[col].permute(1, 2, 0).numpy() * 0.5 + 0.5
            axes[row, col].imshow(img.clip(0, 1))
            axes[row, col].axis("off")
            if col == 0:
                axes[row, col].set_ylabel(labels[row], fontsize=12,
                                           rotation=0, labelpad=80, va='center')
    plt.suptitle(direction, fontsize=14, y=1.01)
    plt.tight_layout()
    return fig

if snapshots:
    rA, fB, rcA, rB, fA, rcB = snapshots[-1]

    fig1 = show_cycle(rA, fB, rcA, "Sketch → Photo → Sketch (cycle)", n=4)
    fig1.savefig(os.path.join(OUTPUT_DIR, "cycle_A2B2A.png"), dpi=150, bbox_inches="tight")
    plt.show()

    fig2 = show_cycle(rB, fA, rcB, "Photo → Sketch → Photo (cycle)", n=4)
    fig2.savefig(os.path.join(OUTPUT_DIR, "cycle_B2A2B.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 6.3 Training Progression

In [ ]:
# ============================================================
# 6.3 Training Progression (Sketch → Photo over epochs)
# ============================================================
if snapshots:
    n_snaps = len(snapshots)
    step = max(1, n_snaps // 5)
    indices = list(range(0, n_snaps, step))
    if (n_snaps - 1) not in indices:
        indices.append(n_snaps - 1)

    fig, axes = plt.subplots(len(indices), 3, figsize=(10, 3 * len(indices)))
    if len(indices) == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(indices):
        rA, fB, rcA, _, _, _ = snapshots[idx]
        for col, (tensor, label) in enumerate(zip(
            [rA[0], fB[0], rcA[0]],
            ["Sketch (Input)", "Photo (Generated)", "Sketch (Reconstructed)"]
        )):
            arr = tensor.permute(1, 2, 0).numpy() * 0.5 + 0.5
            axes[row, col].imshow(arr.clip(0, 1))
            axes[row, col].axis("off")
            if col == 0:
                axes[row, col].set_ylabel(f"Epoch {idx+1}", fontsize=10)
            if row == 0:
                axes[row, col].set_title(label, fontsize=11)

    plt.suptitle("Training Progression: Sketch → Photo", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "progression.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 6.4 Quantitative Evaluation — SSIM & PSNR

Since CycleGAN is *unpaired*, we cannot compare generated images to ground truth.
Instead, we measure **cycle reconstruction quality**: how well the round-trip
(A → B → A) reconstructs the original input.

- High **SSIM** = good structural preservation through the cycle
- High **PSNR** = low pixel-level distortion in reconstruction

In [ ]:
# ============================================================
# 6.4 SSIM & PSNR on Cycle Reconstruction
# ============================================================

def compute_cycle_metrics(G_forward, G_backward, loader, device, max_batches=30):
    """Compute SSIM and PSNR between original and cycle-reconstructed images."""
    G_forward.eval(); G_backward.eval()
    ssim_scores, psnr_scores = [], []

    with torch.no_grad():
        for batch_idx, real in enumerate(loader):
            if batch_idx >= max_batches:
                break
            real = real.to(device)
            fake = G_forward(real)
            recon = G_backward(fake)

            real_np  = (real.cpu().numpy()  * 0.5 + 0.5).clip(0, 1)
            recon_np = (recon.cpu().numpy() * 0.5 + 0.5).clip(0, 1)

            for j in range(real_np.shape[0]):
                img_r = np.transpose(real_np[j], (1, 2, 0))
                img_c = np.transpose(recon_np[j], (1, 2, 0))
                s = ssim(img_r, img_c, multichannel=True,
                         channel_axis=2, data_range=1.0)
                p = psnr(img_r, img_c, data_range=1.0)
                ssim_scores.append(s)
                psnr_scores.append(p)

    G_forward.train(); G_backward.train()
    return np.array(ssim_scores), np.array(psnr_scores)

print("Computing cycle metrics: Sketch → Photo → Sketch")
ssim_A, psnr_A = compute_cycle_metrics(G_AB, G_BA, loader_A, device)

print("Computing cycle metrics: Photo → Sketch → Photo")
ssim_B, psnr_B = compute_cycle_metrics(G_BA, G_AB, loader_B, device)

print(f"\n{'Direction':<30} {'SSIM (mean±std)':>20} {'PSNR dB (mean±std)':>22}")
print("-" * 74)
print(f"{'Sketch→Photo→Sketch':<30} {ssim_A.mean():.4f} ± {ssim_A.std():.4f}"
      f"     {psnr_A.mean():.2f} ± {psnr_A.std():.2f}")
print(f"{'Photo→Sketch→Photo':<30} {ssim_B.mean():.4f} ± {ssim_B.std():.4f}"
      f"     {psnr_B.mean():.2f} ± {psnr_B.std():.2f}")

In [ ]:
# ── Distribution plots ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.hist(ssim_A, bins=25, alpha=0.6, color="tab:blue", label="A→B→A", edgecolor="black")
ax.hist(ssim_B, bins=25, alpha=0.6, color="tab:orange", label="B→A→B", edgecolor="black")
ax.axvline(ssim_A.mean(), color="tab:blue", ls="--", lw=2)
ax.axvline(ssim_B.mean(), color="tab:orange", ls="--", lw=2)
ax.set_title("Cycle SSIM Distribution", fontsize=13)
ax.set_xlabel("SSIM"); ax.set_ylabel("Count"); ax.legend()

ax = axes[1]
ax.hist(psnr_A, bins=25, alpha=0.6, color="tab:green", label="A→B→A", edgecolor="black")
ax.hist(psnr_B, bins=25, alpha=0.6, color="tab:red", label="B→A→B", edgecolor="black")
ax.axvline(psnr_A.mean(), color="tab:green", ls="--", lw=2)
ax.axvline(psnr_B.mean(), color="tab:red", ls="--", lw=2)
ax.set_title("Cycle PSNR Distribution", fontsize=13)
ax.set_xlabel("PSNR (dB)"); ax.set_ylabel("Count"); ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "cycle_ssim_psnr.png"), dpi=150)
plt.show()

# 7. Interactive App Deployment (Gradio)

Upload a sketch **or** a photo and the model will translate it to the other domain in real-time.
A **Cycle** tab shows the full round-trip: Input → Translated → Reconstructed.

In [ ]:
# ============================================================
# 7. Gradio App
# ============================================================
import gradio as gr

transform_infer = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

def tensor_to_pil(tensor):
    arr = tensor.squeeze(0).permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5
    arr = (arr.clip(0, 1) * 255).astype(np.uint8)
    return Image.fromarray(arr)


def translate(image, direction):
    if image is None:
        return None
    img = transform_infer(image.convert("RGB")).unsqueeze(0).to(device)

    G_AB.eval(); G_BA.eval()
    with torch.no_grad():
        if direction == "Sketch → Photo":
            out = G_AB(img)
        else:
            out = G_BA(img)
    G_AB.train(); G_BA.train()
    return tensor_to_pil(out)


def cycle_demo(image, direction):
    if image is None:
        return None, None
    img = transform_infer(image.convert("RGB")).unsqueeze(0).to(device)

    G_AB.eval(); G_BA.eval()
    with torch.no_grad():
        if direction == "Sketch → Photo → Sketch":
            translated = G_AB(img)
            reconstructed = G_BA(translated)
        else:
            translated = G_BA(img)
            reconstructed = G_AB(translated)
    G_AB.train(); G_BA.train()
    return tensor_to_pil(translated), tensor_to_pil(reconstructed)


with gr.Blocks(title="CycleGAN: Sketch ↔ Photo", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## 🔄 CycleGAN — Unpaired Sketch ↔ Photo Translation\n"
        "Upload a sketch or photo to translate it to the other domain. "
        "The Cycle tab demonstrates the round-trip reconstruction."
    )

    with gr.Tab("Translate"):
        with gr.Row():
            with gr.Column():
                inp_img = gr.Image(label="Input Image", type="pil")
                dir_dd  = gr.Dropdown(
                    ["Sketch → Photo", "Photo → Sketch"],
                    value="Sketch → Photo", label="Direction"
                )
                trans_btn = gr.Button("🎨 Translate", variant="primary")
            with gr.Column():
                out_img = gr.Image(label="Translated Output", type="pil")
        trans_btn.click(fn=translate, inputs=[inp_img, dir_dd], outputs=out_img)

    with gr.Tab("Cycle (Round-Trip)"):
        gr.Markdown("See the full cycle: **Input → Translated → Reconstructed**")
        with gr.Row():
            with gr.Column():
                cyc_inp = gr.Image(label="Input Image", type="pil")
                cyc_dir = gr.Dropdown(
                    ["Sketch → Photo → Sketch", "Photo → Sketch → Photo"],
                    value="Sketch → Photo → Sketch", label="Cycle Direction"
                )
                cyc_btn = gr.Button("🔄 Run Cycle", variant="primary")
            with gr.Column():
                cyc_trans = gr.Image(label="Translated", type="pil")
            with gr.Column():
                cyc_recon = gr.Image(label="Reconstructed", type="pil")
        cyc_btn.click(fn=cycle_demo, inputs=[cyc_inp, cyc_dir],
                      outputs=[cyc_trans, cyc_recon])

    with gr.Tab("Validation Gallery"):
        gr.Markdown("Random samples from the training domains.")
        gal_btn = gr.Button("🖼️ Show Samples")
        gallery = gr.Gallery(label="Sketch | Translated Photo | Reconstructed Sketch",
                             columns=3)

        def show_gallery():
            G_AB.eval(); G_BA.eval()
            images = []
            samp_A = next(iter(loader_A))[:5].to(device)
            with torch.no_grad():
                fB = G_AB(samp_A)
                rcA = G_BA(fB)
            for j in range(samp_A.size(0)):
                for t in [samp_A[j], fB[j], rcA[j]]:
                    images.append(tensor_to_pil(t.unsqueeze(0)))
            G_AB.train(); G_BA.train()
            return images

        gal_btn.click(fn=show_gallery, outputs=gallery)

try:
    demo.launch(share=True, inline=False)
except Exception as e:
    print(f"Gradio launch failed: {e}")
    print("Fallback: demo.launch(inline=True)")

# 8. Summary & Conclusions

## How CycleGAN Differs from Pix2Pix

| Aspect | Pix2Pix | CycleGAN |
|--------|---------|----------|
| **Data** | Paired (input ↔ target) | Unpaired (two separate collections) |
| **Generators** | 1 (U-Net) | 2 (ResNet: G_AB + G_BA) |
| **Discriminators** | 1 (PatchGAN on pair) | 2 (PatchGAN on single images) |
| **Key constraint** | L1 reconstruction | Cycle consistency |
| **Use case** | Supervised translation | Domain adaptation without alignment |

## CycleGAN Loss Components

| Loss | Purpose | Effect of Removing It |
|------|---------|----------------------|
| **Adversarial** | Make translated images look real | Outputs become blurry/unrealistic |
| **Cycle Consistency** | Preserve structure through round-trip | Model ignores input structure entirely |
| **Identity** | Preserve colour/tone when input is already in target domain | Colour shifts in output |

## Key Design Decisions

| Choice | Rationale |
|--------|-----------|
| **ResNet generator (6 blocks)** | Better for style transfer than U-Net; 6 blocks fit Kaggle T4 memory |
| **InstanceNorm** | Works better than BatchNorm for style transfer (normalises per-sample) |
| **LSGAN (MSE loss)** | Smoother gradients and more stable training than BCE |
| **Replay buffer (50 images)** | Prevents discriminator from forgetting old fakes; stabilises training |
| **LR linear decay** | Prevents mode collapse in later epochs by gradually reducing step size |
| **Image size 128×128** | Fits within dual T4 GPU memory at batch size 4 |

## Files Saved
- `cyclegan_G_AB_final.pth` / `cyclegan_G_BA_final.pth` — Generator weights
- `cyclegan_D_A_final.pth` / `cyclegan_D_B_final.pth` — Discriminator weights
- `loss_curves.png` — All training loss plots
- `cycle_A2B2A.png` / `cycle_B2A2B.png` — Cycle translation examples
- `progression.png` — Training evolution
- `cycle_ssim_psnr.png` — Quantitative metric distributions